# 线性回归的简洁实现
:label:`sec_linear_concise`

在过去的几年里，出于对深度学习强烈的兴趣，
许多公司、学者和业余爱好者开发了各种成熟的开源框架。
这些框架可以自动化基于梯度的学习算法中重复性的工作。
在 :numref:`sec_linear_scratch`中，我们只运用了：
（1）通过张量来进行数据存储和线性代数；
（2）通过自动微分来计算梯度。
实际上，由于数据迭代器、损失函数、优化器和神经网络层很常用，
现代深度学习库也为我们实现了这些组件。

本节将介绍如何(**通过使用深度学习框架来简洁地实现**)
 :numref:`sec_linear_scratch`中的(**线性回归模型**)。

## 生成数据集

与 :numref:`sec_linear_scratch`中类似，我们首先[**生成数据集**]。


In [1]:
import numpy as np
import torch
from torch.utils import data
from d2l import torch as d2l

/home/dingxingyu/miniconda3/envs/pytorch_new/lib/python3.9/site-packages/torch/cuda/__init__.py:829: UserWarning: Can't initialize NVML
  warnings.warn("Can't initialize NVML")


In [2]:
true_w = torch.tensor([2, -3.4])
true_b = 4.2
features, labels = d2l.synthetic_data(true_w, true_b, 1000)

## 读取数据集

我们可以[**调用框架中现有的API来读取数据**]。
我们将`features`和`labels`作为API的参数传递，并通过数据迭代器指定`batch_size`。
此外，布尔值`is_train`表示是否希望数据迭代器对象在每个迭代周期内打乱数据。


In [3]:
def load_array(data_arrays, batch_size, is_train=True):  #@save
    """构造一个PyTorch数据迭代器"""
    dataset = data.TensorDataset(*data_arrays)
    return data.DataLoader(dataset, batch_size, shuffle=is_train)

In [4]:
batch_size = 10
data_iter = load_array((features, labels), batch_size)

使用`data_iter`的方式与我们在 :numref:`sec_linear_scratch`中使用`data_iter`函数的方式相同。为了验证是否正常工作，让我们读取并打印第一个小批量样本。
与 :numref:`sec_linear_scratch`不同，这里我们使用`iter`构造Python迭代器，并使用`next`从迭代器中获取第一项。


In [5]:
next(iter(data_iter))

[tensor([[-0.5632, -1.0369],
         [-0.4217, -0.8494],
         [-0.5725, -0.5298],
         [ 1.7476,  0.0861],
         [ 0.0056, -1.1195],
         [ 0.2471, -2.1492],
         [ 1.5804,  1.2858],
         [-0.5222,  0.2781],
         [ 1.9553, -1.1995],
         [ 0.0632, -1.6522]]),
 tensor([[ 6.5898],
         [ 6.2457],
         [ 4.8394],
         [ 7.4002],
         [ 8.0162],
         [12.0072],
         [ 2.9999],
         [ 2.2002],
         [12.1916],
         [ 9.9528]])]

## 定义模型

当我们在 :numref:`sec_linear_scratch`中实现线性回归时，
我们明确定义了模型参数变量，并编写了计算的代码，这样通过基本的线性代数运算得到输出。
但是，如果模型变得更加复杂，且当我们几乎每天都需要实现模型时，自然会想简化这个过程。
这种情况类似于为自己的博客从零开始编写网页。
做一两次是有益的，但如果每个新博客就需要工程师花一个月的时间重新开始编写网页，那并不高效。

对于标准深度学习模型，我们可以[**使用框架的预定义好的层**]。这使我们只需关注使用哪些层来构造模型，而不必关注层的实现细节。
我们首先定义一个模型变量`net`，它是一个`Sequential`类的实例。
`Sequential`类将多个层串联在一起。
当给定输入数据时，`Sequential`实例将数据传入到第一层，
然后将第一层的输出作为第二层的输入，以此类推。
在下面的例子中，我们的模型只包含一个层，因此实际上不需要`Sequential`。
但是由于以后几乎所有的模型都是多层的，在这里使用`Sequential`会让你熟悉“标准的流水线”。

回顾 :numref:`fig_single_neuron`中的单层网络架构，
这一单层被称为*全连接层*（fully-connected layer），
因为它的每一个输入都通过矩阵-向量乘法得到它的每个输出。


在PyTorch中，全连接层在`Linear`类中定义。
值得注意的是，我们将两个参数传递到`nn.Linear`中。
第一个指定输入特征形状，即2，第二个指定输出特征形状，输出特征形状为单个标量，因此为1。


In [6]:
# nn是神经网络的缩写
from torch import nn

net = nn.Sequential(nn.Linear(2, 1))

## (**初始化模型参数**)

在使用`net`之前，我们需要初始化模型参数。
如在线性回归模型中的权重和偏置。
深度学习框架通常有预定义的方法来初始化参数。
在这里，我们指定每个权重参数应该从均值为0、标准差为0.01的正态分布中随机采样，
偏置参数将初始化为零。


正如我们在构造`nn.Linear`时指定输入和输出尺寸一样，
现在我们能直接访问参数以设定它们的初始值。
我们通过`net[0]`选择网络中的第一个图层，
然后使用`weight.data`和`bias.data`方法访问参数。
我们还可以使用替换方法`normal_`和`fill_`来重写参数值。


In [7]:
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

tensor([0.])

## 定义损失函数


[**计算均方误差使用的是`MSELoss`类，也称为平方$L_2$范数**]。
默认情况下，它返回所有样本损失的平均值。


In [8]:
loss = nn.MSELoss()

## 定义优化算法


小批量随机梯度下降算法是一种优化神经网络的标准工具，
PyTorch在`optim`模块中实现了该算法的许多变种。
当我们(**实例化一个`SGD`实例**)时，我们要指定优化的参数
（可通过`net.parameters()`从我们的模型中获得）以及优化算法所需的超参数字典。
小批量随机梯度下降只需要设置`lr`值，这里设置为0.03。


In [9]:
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

## 训练

通过深度学习框架的高级API来实现我们的模型只需要相对较少的代码。
我们不必单独分配参数、不必定义我们的损失函数，也不必手动实现小批量随机梯度下降。
当我们需要更复杂的模型时，高级API的优势将大大增加。
当我们有了所有的基本组件，[**训练过程代码与我们从零开始实现时所做的非常相似**]。

回顾一下：在每个迭代周期里，我们将完整遍历一次数据集（`train_data`），
不停地从中获取一个小批量的输入和相应的标签。
对于每一个小批量，我们会进行以下步骤:

* 通过调用`net(X)`生成预测并计算损失`l`（前向传播）。
* 通过进行反向传播来计算梯度。
* 通过调用优化器来更新模型参数。

为了更好的衡量训练效果，我们计算每个迭代周期后的损失，并打印它来监控训练过程。


In [10]:
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X) ,y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l:f}')

epoch 1, loss 0.000244
epoch 2, loss 0.000103
epoch 3, loss 0.000104


下面我们[**比较生成数据集的真实参数和通过有限数据训练获得的模型参数**]。
要访问参数，我们首先从`net`访问所需的层，然后读取该层的权重和偏置。
正如在从零开始实现中一样，我们估计得到的参数与生成数据的真实参数非常接近。


In [11]:
w = net[0].weight.data
print('w的估计误差：', true_w - w.reshape(true_w.shape))
b = net[0].bias.data
print('b的估计误差：', true_b - b)

w的估计误差： tensor([1.0288e-04, 2.2888e-05])
b的估计误差： tensor([-0.0011])


## 小结


* 我们可以使用PyTorch的高级API更简洁地实现模型。
* 在PyTorch中，`data`模块提供了数据处理工具，`nn`模块定义了大量的神经网络层和常见损失函数。
* 我们可以通过`_`结尾的方法将参数替换，从而初始化参数。


## 练习

1. 如果将小批量的总损失替换为小批量损失的平均值，需要如何更改学习率？
1. 查看深度学习框架文档，它们提供了哪些损失函数和初始化方法？用Huber损失代替原损失，即
    $$l(y,y') = \begin{cases}|y-y'| -\frac{\sigma}{2} & \text{ if } |y-y'| > \sigma \\ \frac{1}{2 \sigma} (y-y')^2 & \text{ 其它情况}\end{cases}$$
1. 如何访问线性回归的梯度？


[Discussions](https://discuss.d2l.ai/t/1781)


## 练习解答

### 练习1：小批量损失平均值 vs 总损失，学习率如何调整？

**答案：不需要调整学习率。**

```
分析：

MSELoss默认行为：

reduction='mean'（默认）→ 返回平均损失
  L = 1/n Σ (y_hat - y)²

reduction='sum' → 返回总损失
  L = Σ (y_hat - y)²

---

关键点：PyTorch优化器已经考虑了这一点！

SGD更新公式：
  w ← w - lr × w.grad

如果用平均损失：
  backward()后：
    w.grad = ∂(平均损失)/∂w = 1/n × Σ ∂l_i/∂w
    → 已经是平均梯度
  
  更新：
    w ← w - lr × 平均梯度

如果用总损失（reduction='sum'）：
  backward()后：
    w.grad = ∂(总损失)/∂w = Σ ∂l_i/∂w
    → 是累加梯度
  
  更新：
    w ← w - lr × 累加梯度
    → 更新幅度更大（约n倍）

---

两种方式比较：

假设batch_size=10：

用平均损失：
  w.grad = 平均梯度（已除以10）
  更新 = lr × 平均梯度
  → 学习率不需要改

用总损失：
  w.grad = 累加梯度（未除以10）
  更新 = lr × 累加梯度
  → 如果用相同lr，更新幅度是平均损失的10倍
  → 需要把lr缩小为原来的1/10

---

验证代码：

# 方式1：平均损失
loss = nn.MSELoss()  # 默认 reduction='mean'
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

# 方式2：总损失
loss = nn.MSELoss(reduction='sum')
trainer = torch.optim.SGD(net.parameters(), lr=0.03/10)  # lr缩小10倍

两种方式效果相同！

---

结论：

如果MSELoss用平均值（默认）：
  → 学习率保持不变

如果MSELoss用总损失：
  → 学习率要缩小为 lr / batch_size

公式对应：
  平均损失：lr × 平均梯度
  总损失：  (lr/batch_size) × 累加梯度 = lr × 平均梯度
```

---

### 练习2：深度学习框架的损失函数和初始化方法

**PyTorch常见损失函数：**

```python
import torch.nn as nn

# 回归问题
nn.MSELoss()       → 均方误差（平方损失）
nn.L1Loss()        → 绝对误差损失
nn.SmoothL1Loss()  → Huber损失（平滑L1）
nn.PoissonLoss()   → 泊松损失

# 分类问题
nn.CrossEntropyLoss()  → 交叉熵损失（多分类）
nn.BCELoss()           → 二元交叉熵损失
nn.BCEWithLogitsLoss() → 带sigmoid的BCE损失
nn.NLLLoss()           → 负对数似然损失
nn.KLDivLoss()         → KL散度损失

# 其他
nn.CosineEmbeddingLoss() → 余弦相似度损失
nn.MarginRankingLoss()   → 排序损失
nn.TripletMarginLoss()   → 三元组损失
```

---

**PyTorch常见初始化方法：**

```python
import torch.nn.init as init

# 正态分布
init.normal_(tensor, mean=0, std=0.01)  → 正态分布初始化
init.randn_(tensor)                     → 标准正态分布

# 均匀分布
init.uniform_(tensor, a=0, b=1)         → 均匀分布
init.rand_(tensor)                      → [0,1]均匀分布

# 常数
init.constant_(tensor, val)             → 常数初始化
init.zeros_(tensor)                     → 零初始化
init.ones_(tensor)                      → 一初始化

# Xavier初始化（考虑输入输出维度）
init.xavier_uniform_(tensor)            → Xavier均匀分布
init.xavier_normal_(tensor)              → Xavier正态分布

# He初始化（针对ReLU）
init.kaiming_uniform_(tensor)           → He均匀分布
init.kaiming_normal_(tensor)            → He正态分布

# 其他
init.orthogonal_(tensor)                → 正交初始化
init.sparse_(tensor, sparsity)          → 稀疏初始化
```

---

**用Huber损失替代MSELoss：**

```python
Huber损失公式：

l(y, y') = |y-y'| - σ/2     if |y-y'| > σ
         = 1/(2σ) (y-y')²   otherwise

特点：
  → 误差大时：用绝对误差（对异常值鲁棒）
  → 误差小时：用平方误差（平滑，可导）
  → σ是阈值参数（默认σ=1）

---

PyTorch实现：

# 使用SmoothL1Loss（就是Huber损失）
loss = nn.SmoothL1Loss()

# 训练代码不变
num_epochs = 3
for epoch in range(num_epochs):
    for X, y in data_iter:
        l = loss(net(X), y)
        trainer.zero_grad()
        l.backward()
        trainer.step()
    l = loss(net(features), labels)
    print(f'epoch {epoch + 1}, loss {l:f}')

---

对比MSELoss：

MSELoss：
  → 所有误差都用平方（大误差惩罚重）
  → 对异常值敏感

SmoothL1Loss（Huber）：
  → 大误差用绝对值（惩罚轻）
  → 对异常值更鲁棒
  → 在0点附近平滑（可导）

应用场景：
  → 数据有异常值 → 用Huber更好
  → 数据干净 → MSE更好
```

---

### 练习3：如何访问线性回归的梯度？

**方法：通过net[0].weight.grad 和 net[0].bias.grad**

```python
# 创建模型
net = nn.Sequential(nn.Linear(2, 1))

# 初始化参数
net[0].weight.data.normal_(0, 0.01)
net[0].bias.data.fill_(0)

# 定义损失和优化器
loss = nn.MSELoss()
trainer = torch.optim.SGD(net.parameters(), lr=0.03)

# 训练一步
for X, y in data_iter:
    l = loss(net(X), y)
    trainer.zero_grad()
    l.backward()
    
    # 访问梯度（在backward之后，step之前）
    weight_grad = net[0].weight.grad
    bias_grad = net[0].bias.grad
    
    print(f'权重梯度: {weight_grad}')
    print(f'偏置梯度: {bias_grad}')
    
    trainer.step()
    break

---

输出示例：
权重梯度: tensor([[-0.1234, 0.5678]])
偏置梯度: tensor([-0.0234])

解读：
  weight_grad.shape = (1, 2) ← PyTorch权重形状
  bias_grad.shape = (1)

---

层级结构：

net（Sequential）
  │
  ├── net[0]（Linear层）
  │     │
  │     ├── weight（Parameter）
  │     │     │
  │     │     ├── .data → 权重值
  │     │     └── .grad → 权重梯度
  │     │
  │     └── bias（Parameter）
  │           │
  │           ├── .data → 偏置值
  │           └── .grad → 偏置梯度

---

完整访问方式：

# 访问权重值
net[0].weight.data

# 访问权重梯度
net[0].weight.grad

# 访问偏置值
net[0].bias.data

# 访问偏置梯度
net[0].bias.grad

# 查看梯度是否为None
print(net[0].weight.grad is None)  # True（backward前）
print(net[0].weight.grad is None)  # False（backward后）
```

---

### 一句话总结

| 练习 | 核心答案 |
|------|---------|
| 1. 平均损失vs总损失 | 默认平均不需要改；用总损失需lr缩小为1/batch_size |
| 2. Huber损失替代 | 用 nn.SmoothL1Loss()，对异常值更鲁棒 |
| 3. 访问梯度 | net[0].weight.grad 和 net[0].bias.grad |